# QLoRA Fine-Tuning Gemma-4-2B on Medical Q&A

## 1. Setup and Installation

In [ ]:
!pip install transformers peft bitsandbytes accelerate datasets scipy unsloth torch -q

## 2. Load Processed Data from S3

In [ ]:
BUCKET_NAME = 'q1abc-ai-medical'
DATA_PREFIX = 'processed/'

## 3. Load and Prepare Dataset

In [ ]:
from datasets import Dataset

# Convert to HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)

In [ ]:
# Display a sample prompt
sample = train_dataset[0]
print("Sample prompt:")
print(sample['prompt'])

## 4. Load Gemma Model with QLoRA Configuration

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'google/gemma-4-2b'
MAX_SEQ_LENGTH = 1024

## 5. Training Configuration

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./gemma-qlora-finetuned-medical',
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    warmup_steps=50,
    fp16=True,
    eval_strategy='steps',
    save_strategy='steps',
    load_best_model_at_end=True,
    max_seq_length=1024,  # Changed from 512
)

## 6. Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    """Tokenize prompts and prepare labels."""
    result = tokenizer(
        examples['prompt'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding='longest'
    )
    # Labels are the same as input_ids for causal LM
    result['labels'] = result['input_ids'].copy()
    return result

## 7. Train Model

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not masked LM
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Train
trainer.train()

# Save model
trainer.save_model('./gemma-qlora-finetuned-final')

## 8. Export to GGUF for Ollama

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer

# Reload base model and merge LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto'
)

model = PeftModel.from_pretrained(base_model, './gemma-qlora-finetuned-final')
model = model.merge_and_unload()

# Save as HuggingFace format
model.save_pretrained('./gemma-qlora-gguf-medical')
tokenizer.save_pretrained('./gemma-qlora-gguf-medical')

print("Model exported to ./gemma-qlora-gguf-medical")
print("To convert to GGUF for Ollama, use: llama-cli or llama.cpp converter")

## 9. Inference Comparison: Base vs Fine-tuned on Medical Q&A

In [ ]:
# Reload base model for comparison
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto'
)

fine_tuned_model = trainer.model

test_prompts = [
    '### Medical Question: What is the mechanism of action of ibuprofen?\n\n### Clinical Context:\n\n### Answer:',
    '### Medical Question: Describe the pathophysiology of type 2 diabetes\n\n### Clinical Context:\n\n### Answer:',
]